# Tạo 500 Preference Data: SFT vs RL/PPO bằng BERTScore

Notebook này tạo file preference data từ tập test trên Google Drive.

Mỗi dòng output có format:

```json
{
  "id": "pref_000001",
  "source_id": "...",
  "image": "...",
  "question": "...",
  "reference_answer": "...",
  "sft_answer": "...",
  "rl_answer": "...",
  "sft_bertscore_f1": 0.88,
  "rl_bertscore_f1": 0.95,
  "auto_preference": "rl",
  "auto_preference_margin": 0.07,
  "auto_preference_method": "bertscore_f1_only",
  "human_eval": {
    "preference": null,
    "comment": ""
  }
}
```

Logic chọn preference:

```text
sft_score = BERTScore-F1(sft_answer, reference_answer)
rl_score  = BERTScore-F1(rl_answer, reference_answer)

Nếu |rl_score - sft_score| < TIE_THRESHOLD -> tie
Nếu rl_score > sft_score -> rl
Ngược lại -> sft
```

## 1. Cài thư viện

In [ ]:
!pip -q install -U transformers accelerate peft bitsandbytes qwen-vl-utils pillow tqdm pandas bert-score

In [ ]:
import gc
import json
import random
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Cấu hình đường dẫn Drive

Nếu tên folder trên Drive khác ảnh chụp, sửa các biến trong cell dưới.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning')

# Test JSONL trên Drive.
TEST_JSONL = PROJECT_ROOT / 'Split' / 'test' / 'manual_test.jsonl'

# Ảnh test. Notebook sẽ thử nhiều root để khớp với dữ liệu image trong JSONL.
IMAGE_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'Split' / 'test',
    PROJECT_ROOT / '_images_final_unzipped' / 'images_final',
    PROJECT_ROOT / '_images_final_unzipped',
]

# Adapter SFT sau fine-tune QLoRA.
SFT_ADAPTER_PATH = PROJECT_ROOT / 'qwen25_vl_herb_qlora_10chunks' / 'final_finetuned_adapter'

# Adapter RL/PPO sau PPO.
PPO_ADAPTER_PATH = PROJECT_ROOT / 'PPO' / 'qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume' / 'final_ppo_adapter'

# Nếu folder PPO của bạn khác, sửa dòng trên. Ví dụ:
# PPO_ADAPTER_PATH = PROJECT_ROOT / 'PPO' / 'ten_folder_ppo_cua_ban' / 'final_ppo_adapter'

OUTPUT_DIR = PROJECT_ROOT / 'preference_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = OUTPUT_DIR / 'preference_500_bertscore_sft_vs_rl.jsonl'
OUTPUT_CSV = OUTPUT_DIR / 'preference_500_bertscore_sft_vs_rl.csv'

assert TEST_JSONL.exists(), f'Không thấy TEST_JSONL: {TEST_JSONL}'
assert SFT_ADAPTER_PATH.exists(), f'Không thấy SFT adapter: {SFT_ADAPTER_PATH}'
assert PPO_ADAPTER_PATH.exists(), f'Không thấy PPO adapter: {PPO_ADAPTER_PATH}'

print('TEST_JSONL:', TEST_JSONL)
print('SFT_ADAPTER_PATH:', SFT_ADAPTER_PATH)
print('PPO_ADAPTER_PATH:', PPO_ADAPTER_PATH)
print('OUTPUT_JSONL:', OUTPUT_JSONL)

## 3. Đọc 500 mẫu test

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

all_test_rows = read_jsonl(TEST_JSONL)
print('Total test rows:', len(all_test_rows))
all_test_rows[:2]

In [ ]:
NUM_PREF = 500
SELECTION_MODE = 'first'  # 'first' hoặc 'random'
RANDOM_SEED = 42

if SELECTION_MODE == 'random':
    rng = random.Random(RANDOM_SEED)
    test_rows = rng.sample(all_test_rows, min(NUM_PREF, len(all_test_rows)))
else:
    test_rows = all_test_rows[:NUM_PREF]

print('Selected rows:', len(test_rows))

## 4. Hàm resolve ảnh

Dataset có thể lưu `image` dạng `images/Pxxxx.jpg`. Cell này sẽ tự thử các root phổ biến.

In [ ]:
def resolve_image_path(image_value):
    image_value = str(image_value).replace('\\\\', '/').replace('\\', '/')
    candidates = []
    for root in IMAGE_ROOT_CANDIDATES:
        candidates.append(root / image_value)
        candidates.append(root / Path(image_value).name)

    # Một số cấu trúc Drive hay gặp.
    candidates.append(PROJECT_ROOT / 'Split' / 'test' / image_value)
    candidates.append(PROJECT_ROOT / 'Split' / 'test' / 'images' / Path(image_value).name)
    candidates.append(PROJECT_ROOT / '_images_final_unzipped' / 'images_final' / Path(image_value).name)

    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Không tìm thấy ảnh cho image={image_value}. Thử candidates đầu: {candidates[:5]}')

# Test nhanh vài ảnh.
for row in test_rows[:5]:
    print(row['image'], '->', resolve_image_path(row['image']))

## 5. Load Qwen2.5-VL + adapter

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

def load_model_with_adapter(adapter_path):
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
    )
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    return model

## 6. Hàm inference

In [ ]:
def build_prompt(question):
    return (
        'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
        'Hãy trả lời câu hỏi bằng tiếng Việt, thật ngắn gọn, tối đa 10 từ. '
        'Không giải thích thêm.\n'
        f'Câu hỏi: {question}'
    )

@torch.inference_mode()
def predict_one(model, row, max_new_tokens=32):
    image_path = resolve_image_path(row['image'])
    image = Image.open(image_path).convert('RGB')

    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': build_prompt(row['question'])},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    if torch.cuda.is_available():
        inputs = inputs.to('cuda')

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    answer = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return answer.strip()

## 7. Sinh câu trả lời SFT

In [ ]:
sft_pred_path = OUTPUT_DIR / 'sft_predictions_for_preference_500.jsonl'

done = {}
if sft_pred_path.exists():
    for row in read_jsonl(sft_pred_path):
        done[row['source_id']] = row
print('Existing SFT predictions:', len(done))

sft_model = load_model_with_adapter(SFT_ADAPTER_PATH)

with open(sft_pred_path, 'a', encoding='utf-8') as f:
    for i, row in enumerate(tqdm(test_rows), start=1):
        source_id = row.get('id') or f'source_{i:06d}'
        if source_id in done:
            continue
        try:
            pred = predict_one(sft_model, row)
            error = ''
        except Exception as e:
            pred = ''
            error = repr(e)
        out = {
            'source_id': source_id,
            'image': row['image'],
            'question': row['question'],
            'reference_answer': row['answer'],
            'sft_answer': pred,
            'error': error,
        }
        f.write(json.dumps(out, ensure_ascii=False) + '\n')
        f.flush()

del sft_model
gc.collect()
torch.cuda.empty_cache()
print('Saved:', sft_pred_path)

## 8. Sinh câu trả lời RL/PPO

In [ ]:
rl_pred_path = OUTPUT_DIR / 'rl_predictions_for_preference_500.jsonl'

done = {}
if rl_pred_path.exists():
    for row in read_jsonl(rl_pred_path):
        done[row['source_id']] = row
print('Existing RL predictions:', len(done))

rl_model = load_model_with_adapter(PPO_ADAPTER_PATH)

with open(rl_pred_path, 'a', encoding='utf-8') as f:
    for i, row in enumerate(tqdm(test_rows), start=1):
        source_id = row.get('id') or f'source_{i:06d}'
        if source_id in done:
            continue
        try:
            pred = predict_one(rl_model, row)
            error = ''
        except Exception as e:
            pred = ''
            error = repr(e)
        out = {
            'source_id': source_id,
            'image': row['image'],
            'question': row['question'],
            'reference_answer': row['answer'],
            'rl_answer': pred,
            'error': error,
        }
        f.write(json.dumps(out, ensure_ascii=False) + '\n')
        f.flush()

del rl_model
gc.collect()
torch.cuda.empty_cache()
print('Saved:', rl_pred_path)

## 9. Tính BERTScore-F1 và tạo preference JSONL

In [ ]:
from bert_score import score as bert_score

sft_rows = {r['source_id']: r for r in read_jsonl(sft_pred_path)}
rl_rows = {r['source_id']: r for r in read_jsonl(rl_pred_path)}

pairs = []
for i, row in enumerate(test_rows, start=1):
    source_id = row.get('id') or f'source_{i:06d}'
    if source_id not in sft_rows or source_id not in rl_rows:
        continue
    pairs.append({
        'source_id': source_id,
        'image': row['image'].replace('\\\\', '/').replace('\\', '/'),
        'question': row['question'],
        'reference_answer': row['answer'],
        'sft_answer': sft_rows[source_id].get('sft_answer', ''),
        'rl_answer': rl_rows[source_id].get('rl_answer', ''),
    })

print('Pairs:', len(pairs))

In [ ]:
refs = [p['reference_answer'] for p in pairs]
sft_preds = [p['sft_answer'] for p in pairs]
rl_preds = [p['rl_answer'] for p in pairs]

_, _, sft_f1 = bert_score(
    sft_preds,
    refs,
    lang='vi',
    model_type='xlm-roberta-base',
    verbose=True,
    rescale_with_baseline=False,
)
_, _, rl_f1 = bert_score(
    rl_preds,
    refs,
    lang='vi',
    model_type='xlm-roberta-base',
    verbose=True,
    rescale_with_baseline=False,
)

sft_f1 = sft_f1.cpu().tolist()
rl_f1 = rl_f1.cpu().tolist()

In [ ]:
TIE_THRESHOLD = 0.01

pref_rows = []
for i, p in enumerate(pairs, start=1):
    sft_score = float(sft_f1[i - 1])
    rl_score = float(rl_f1[i - 1])
    diff = rl_score - sft_score

    if abs(diff) < TIE_THRESHOLD:
        pref = 'tie'
        margin = abs(diff)
    elif diff > 0:
        pref = 'rl'
        margin = diff
    else:
        pref = 'sft'
        margin = -diff

    pref_rows.append({
        'id': f'pref_{i:06d}',
        'source_id': p['source_id'],
        'image': p['image'],
        'question': p['question'],
        'reference_answer': p['reference_answer'],
        'sft_answer': p['sft_answer'],
        'rl_answer': p['rl_answer'],
        'sft_bertscore_f1': round(sft_score, 6),
        'rl_bertscore_f1': round(rl_score, 6),
        'auto_preference': pref,
        'auto_preference_margin': round(float(margin), 6),
        'auto_preference_method': 'bertscore_f1_only',
        'human_eval': {
            'preference': None,
            'comment': ''
        }
    })

with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for row in pref_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

pd.DataFrame(pref_rows).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print('Saved JSONL:', OUTPUT_JSONL)
print('Saved CSV:', OUTPUT_CSV)
pd.Series([r['auto_preference'] for r in pref_rows]).value_counts()

## 10. Xem thử dữ liệu preference

In [ ]:
pref_rows[:3]